# Initialze the Carla client

Must run carla first before a connection can be made

## Install packages

In [12]:
#!python3 -m pip install --upgrade pip
#!python3 -m pip install carla 

import carla

ModuleNotFoundError: No module named 'carla'

# Direct vehicle test  

In [8]:
import carla
import pygame
import math
import asyncio
import socket, json 
from pygame.locals import K_w, K_s, K_a, K_d, K_SPACE, K_ESCAPE

async def get_configR(reader, writer):

        # Recieve data strip
        data = await reader.read(4096)
        data = data.decode().strip()

        # Print recieved data
        print("Java:", data)

        # Load json String data 
        vehicle = json.loads(data)["Vehicle"]
        town = json.loads(data)["town"]
        coordinate_x = json.loads(data)["coordinates"]["x"]
        coordinate_y = json.loads(data)["coordinates"]["y"]
        coordinate_z = json.loads(data)["coordinates"]["z"]
        jType = json.loads(data)["type"]
        #cmd = json.loads(data)["command"]
    
        print(vehicle)
        print(town)
        print(coordinate_x)
        print(jType)

        # Send python reply
        #reply = {"response": f"Python recieved config: {json.loads(data)}"}

        #writer.write((json.dumps(reply) + "\n").encode())
        #await writer.drain()


        return {
            "type": jType,
            "coordinate_x": coordinate_x,
            "coordinate_y": coordinate_y,
            "coordinate_z": coordinate_z,
            "town": town,
            "vehicle": vehicle,
        }


# Initialize any global variablesabs
global collision_data
collision_data = None

# --- Initialize Pygame for keyboard input ---
pygame.init()
screen = pygame.display.set_mode((400, 300))
pygame.display.set_caption("Carla Bike Control")

# --- Connect to Carla simulator ---
client = carla.Client('localhost', 2000)
client.set_timeout(50.0)

# Initializing socket to Java
reader, writer = await asyncio.open_connection("127.0.0.1", 9000)

# Read in object configurations from Java
ob = await get_configR(reader, writer)
print(ob["type"])
print(ob["coordinate_x"])

# load world


client.load_world(ob["town"])
world = client.get_world()

# Destroy vehicle before start of simulator
# OPTIONAL

# Get spectator location
spectator = world.get_spectator()

new_loc = carla.Location(
    x=ob["coordinate_x"],
    y=ob["coordinate_y"],
    z=ob["coordinate_z"]
)

#I'm guessing where the spectator faces
new_rot = carla.Rotation(pitch=0, yaw=0, roll=0)

spectator.set_transform(carla.Transform(new_loc, new_rot))


# --- Choose vehicle blueprint ---
vehicle_bp = world.get_blueprint_library().find(ob["vehicle"])


# --- Use the same location for spawn point ---
spawn_point = carla.Transform(new_loc, new_rot)

vehicle = world.spawn_actor(vehicle_bp, spawn_point)


# Attach collision sensor 
collision_bp = world.get_blueprint_library().find('sensor.other.collision')
collision_transform = carla.Transform(vehicle.get_transform().location)
collision_sensor = world.spawn_actor(collision_bp, collision_transform, attach_to=vehicle)





# Set up actions taken after collsision has been done 
def on_collision(event):
    global collision_data
    
    impulse = event.normal_impulse
    magnitude = math.sqrt(impulse.x**2 + impulse.y**2 + impulse.z**2)

    print("Did the collision get detected?")
    collision_data = {
        "type": "collision",
        "town": ob["town"],
        "actor": event.actor.type_id,
        "time": event.timestamp,
        "impact_force": magnitude
    }





# Send json over to Java socket
async def send_json(writer, data):
    
    # Encode JSON + newline (Java friendly)
    message = json.dumps(data) + "\n"
    writer.write(message.encode("utf-8"))
    await writer.drain()

    print("Sent:", message)




#TODO: write some functions that represent specific types of telemetery data turned into JSON to be recieved. 


collision_sensor.listen(on_collision)

# Main control loop
try:
    control = carla.VehicleControl()
    clock = pygame.time.Clock()
    
    running = True
    while running:
        clock.tick(30)  # 30 FPS
        
        # Reset controls each frame
        control.throttle = 0.0
        control.brake = 0.0
        control.steer = 0.0
        control.hand_brake = False
        
        # Check pressed keys
        keys = pygame.key.get_pressed()
        if keys[K_w]:
            control.throttle = 1
        if keys[K_s]:
            control.brake = 0.5
        if keys[K_a]:
            control.steer = -0.5
        if keys[K_d]:
            control.steer = 0.5
        if keys[K_SPACE]:
            control.hand_brake = True
        
        vehicle.apply_control(control)

        #Build json to send over
        data = {
            "type": "telemetry",
            "town": ob["town"],
            "throttle": control.throttle,
            "steer": control.steer,
            "brake": control.brake,
            "speed": vehicle.get_velocity().length(),
            "location": {
                "x": vehicle.get_location().x,
                "y": vehicle.get_location().y,
                "z": vehicle.get_location().z
            }
        }

        #print("Current location: ", spectator.location)

        if collision_data is not None:
            await send_json(writer, collision_data)
            print("Sent collision:", collision_data)
            collision_data = None
        
        # Send data across socket 
        await send_json(writer, data)
        
        
        # Exit on ESC
        for event in pygame.event.get():
            if event.type == pygame.QUIT or keys[K_ESCAPE]:
                running = False
                writer.close()
                await writer.wait_closed()
                

# Clean up
finally:
    vehicle.destroy()
    pygame.quit()
    print("Vehicle destroyed, script ended.")

ModuleNotFoundError: No module named 'carla'

# Socket management functions

In [2]:
import socket
import json

def get_config(s):

    # Keep loop going 
    while True:

        # Recieve data strip
        data = s.recv(4096).decode().strip()
        if not data:
            break

        # Print recieved data
        print("Java:", data)

        # Load json String data 
        vehicle = json.loads(data)["Vehicle"]
        town = json.loads(data)["town"]
        coordinate_x = json.loads(data)["coordinates"]["x"]
        coordinate_y = json.loads(data)["coordinates"]["y"]
        coordinate_z = json.loads(data)["coordinates"]["z"]
        jType = json.loads(data)["type"]
        #cmd = json.loads(data)["command"]
    
        print(vehicle)
        print(town)
        print(coordinate_x)
        print(jType)

        # Send python reply
        #reply = {"response": f"Python recieved config: {json.loads(data)}"}

        #s.sendall((json.dumps(reply) + "\n").encode())

    return {
        "type": jType,
        "coordinate_x": coordinate_x,
        "coordinate_y": coordinate_y,
        "coordinate_z": coordinate_z,
        "town": town,
        "vehicle": vehicle,
    }

async def get_configR(reader, writer):

        # Recieve data strip
        data = await reader.read(4096)
        data = data.decode().strip()

        # Print recieved data
        print("Java:", data)

        # Load json String data 
        vehicle = json.loads(data)["Vehicle"]
        town = json.loads(data)["town"]
        coordinate_x = json.loads(data)["coordinates"]["x"]
        coordinate_y = json.loads(data)["coordinates"]["y"]
        coordinate_z = json.loads(data)["coordinates"]["z"]
        jType = json.loads(data)["type"]
        #cmd = json.loads(data)["command"]
    
        print(vehicle)
        print(town)
        print(coordinate_x)
        print(jType)

        # Send python reply
        #reply = {"response": f"Python recieved config: {json.loads(data)}"}

        #writer.write((json.dumps(reply) + "\n").encode())
        #await writer.drain()


        return {
            "type": jType,
            "coordinate_x": coordinate_x,
            "coordinate_y": coordinate_y,
            "coordinate_z": coordinate_z,
            "town": town,
            "vehicle": vehicle,
        }



# Autonomous Vehicle Test

In [3]:
import carla
import asyncio
import json
import math
import socket
from agents.navigation.behavior_agent import BehaviorAgent

# --- ASYNC SOCKET SETUP ---
reader, writer = await asyncio.open_connection("127.0.0.1", 9000)


async def send_json(writer, data):
    msg = json.dumps(data) + "\n"
    writer.write(msg.encode("utf-8"))
    await writer.drain()


# --- CONNECT TO CARLA ---
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
world = client.get_world()

# Spawn ego vehicle
blueprint = world.get_blueprint_library().filter("vehicle.tesla.model3")[0]
spawn_point = world.get_map().get_spawn_points()[0]
vehicle = world.spawn_actor(blueprint, spawn_point)

# Create agent (autonomous driver)
agent = BehaviorAgent(vehicle, behavior="normal")

# Set a destination
destination = world.get_map().get_spawn_points()[10].location
agent.set_destination(destination)

# --- ATTACH COLLISION SENSOR ---
collision_bp = world.get_blueprint_library().find("sensor.other.collision")
collision = world.spawn_actor(collision_bp, carla.Transform(), attach_to=vehicle)

async def on_collision(event):
    impulse = event.normal_impulse
    force = math.sqrt(impulse.x**2 + impulse.y**2 + impulse.z**2)

    collision_data = {
        "event": "collision",
        "force": force,
        "other_actor": event.other_actor.type_id,
        "timestamp": world.get_snapshot().timestamp.elapsed_seconds
    }

    await send_json(writer, collision_data)

collision.listen(lambda e: asyncio.ensure_future(on_collision(e)))

# --- ATTACH LANE INVASION SENSOR ---
lane_bp = world.get_blueprint_library().find("sensor.other.lane_invasion")
lane_sensor = world.spawn_actor(lane_bp, carla.Transform(), attach_to=vehicle)

async def on_lane_invasion(event):
    d = {
        "event": "lane_invasion",
        "timestamp": world.get_snapshot().timestamp.elapsed_seconds,
        "crossed": [str(x) for x in event.crossed_lane_markings]
    }
    await send_json(writer, d)

lane_sensor.listen(lambda e: asyncio.ensure_future(on_lane_invasion(e)))


# --- MAIN LOOP ---
try:
    while True:
        # Let the agent compute control
        control = agent.run_step()
        vehicle.apply_control(control)

        # Telemetry packet
        telemetry = {
            "type": "telemetry"
            "throttle": control.throttle,
            "steer": control.steer,
            "brake": control.brake,
            "speed": vehicle.get_velocity().length(),
            "location": {
                "x": vehicle.get_location().x,
                "y": vehicle.get_location().y,
                "z": vehicle.get_location().z
            }
        }

        await send_json(writer, telemetry)
        await asyncio.sleep(0.03)  # 30Hz

finally:
    vehicle.destroy()


SyntaxError: invalid syntax (2543652868.py, line 79)

# Table of Vehicles

In [3]:
import carla

# --- Connect to Carla simulator ---
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

world = client.get_world()
blueprints = world.get_blueprint_library().filter("vehicle.*")

for bp in blueprints:
    print(bp.id)


vehicle.audi.a2
vehicle.nissan.micra
vehicle.audi.tt
vehicle.mercedes.coupe_2020
vehicle.bmw.grandtourer
vehicle.harley-davidson.low_rider
vehicle.ford.ambulance
vehicle.micro.microlino
vehicle.carlamotors.firetruck
vehicle.carlamotors.carlacola
vehicle.carlamotors.european_hgv
vehicle.ford.mustang
vehicle.chevrolet.impala
vehicle.lincoln.mkz_2020
vehicle.citroen.c3
vehicle.dodge.charger_police
vehicle.nissan.patrol
vehicle.jeep.wrangler_rubicon
vehicle.mini.cooper_s
vehicle.mercedes.coupe
vehicle.dodge.charger_2020
vehicle.ford.crown
vehicle.seat.leon
vehicle.toyota.prius
vehicle.yamaha.yzf
vehicle.kawasaki.ninja
vehicle.bh.crossbike
vehicle.mitsubishi.fusorosa
vehicle.tesla.model3
vehicle.gazelle.omafiets
vehicle.tesla.cybertruck
vehicle.diamondback.century
vehicle.mercedes.sprinter
vehicle.audi.etron
vehicle.volkswagen.t2
vehicle.lincoln.mkz_2017
vehicle.dodge.charger_police_2020
vehicle.vespa.zx125
vehicle.mini.cooper_s_2021
vehicle.nissan.patrol_2021
vehicle.volkswagen.t2_2021


In [13]:
import carla

client = carla.Client('localhost', 2000)
client.set_timeout(30.0)

# Load the desired town/map
world = client.load_world("Town10HD")

# Optional: get the map
carla_map = world.get_map()
print("Current map:", carla_map.name)

Current map: Carla/Maps/Town10HD


In [11]:
import carla

# Connect to the CARLA server
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

# Get list of available maps
available_maps = client.get_available_maps()  # returns a list of strings

print("Available maps in CARLA:")
for town in available_maps:
    print(town)

Available maps in CARLA:
Town01
Town01_Opt
Town02
Town02_Opt
Town03
Town03_Opt
Town04
Town04_Opt
Town05
Town05_Opt
Town10HD
Town10HD_Opt
AnnotationColorLandscape
